<a href="https://colab.research.google.com/github/RaviduSenavirathna/Sinhala-Word-Recognizer/blob/main/dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
# Mount Google Drive to access files, if needed.
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import cv2
import numpy as np
import random

# =========================
# CONFIGURATION
# =========================
ORIGINAL_DIR = "/content/drive/MyDrive/Sinhala-CNN/original"
AUGMENTED_DIR = "/content/drive/MyDrive/Sinhala-CNN/augmented"
PROCESSED_DIR = "/content/drive/MyDrive/Sinhala-CNN/processed"
TARGET_PER_CLASS = 150
IMG_SIZE = 128

# Define the crop funtion
def tight_crop(img):
    ys, xs = np.where(img > 0)

    # If no foreground pixels are found (entirely black image), return the original image
    if len(ys) == 0 or len(xs) == 0:
        return img

    # Calculate crop boundaries, ensuring they are within image bounds
    h, w = img.shape[:2]
    y_min = max(0, ys.min() - 2)
    y_max = min(h, ys.max() + 1)
    x_min = max(0, xs.min() - 1)
    x_max = min(w, xs.max() + 2)

    # Ensure valid slice (start < end) to prevent empty arrays
    if y_min >= y_max or x_min >= x_max:
        return img # If crop results in an empty region, return the original image

    return img[y_min:y_max, x_min:x_max]

# Define the image preprocessing function
def img_processing(img):
    # Convert image to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Apply Gaussian blur to reduce noise
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    # Apply Otsu's thresholding to convert to binary image
    _, thresh = cv2.threshold(
        blur,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )
    return thresh

In [3]:
def random_rotation(image):
    # Rotate the image by a small, random angle to simulate slight hand variations
    angle = random.uniform(-5, 5)  # small realistic rotation range
    h, w = image.shape[:2]
    # Get the rotation matrix
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    # Apply the rotation, filling borders with white (255)
    rotated = cv2.warpAffine(image, M, (w, h), borderValue=(255,255,255))
    return rotated

def random_ink_density(image):
    # Simulate ink variation by adjusting contrast and brightness
    alpha = random.uniform(0.6, 1.4)  # contrast adjustment factor
    beta = random.randint(-10, 10)    # brightness adjustment value
    # Apply contrast and brightness changes
    adjusted = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
    return adjusted

def slight_thickness_variation(image):
    # Simulate slight variations in stroke thickness using dilation or erosion
    kernel_size = random.choice([1,2]) # Randomly choose a small kernel size
    kernel = np.ones((kernel_size,kernel_size), np.uint8)

    if random.random() > 0.5:
        # Dilate (thicken) the lines
        image = cv2.dilate(image, kernel, iterations=1)
    else:
        # Erode (thin) the lines
        image = cv2.erode(image, kernel, iterations=1)
    return image

In [4]:
# Create the augmented dataset directory if it doesn't exist
os.makedirs(AUGMENTED_DIR, exist_ok=True)

# Iterate through each class (subfolder) in the original dataset directory
for class_name in os.listdir(ORIGINAL_DIR):
    class_path = os.path.join(ORIGINAL_DIR, class_name)

    # Skip if it's not a directory
    if not os.path.isdir(class_path):
        continue

    print(f"\nProcessing class: {class_name}")

    # Create the output directory for the current class in the augmented dataset
    output_class_path = os.path.join(AUGMENTED_DIR, class_name)
    os.makedirs(output_class_path, exist_ok=True)

    # Get all PNG images in the current class folder
    images = [f for f in os.listdir(class_path)
              if f.endswith('.png')]

    # If no images are found, skip this class
    if len(images) == 0:
        print(f"⚠ No images found in {class_name}. Skipping...")
        continue

    count = 0
    original_imgs = []

    # Load original images and save them to the augmented directory
    for img_name in images:
        img_path = os.path.join(class_path, img_name)
        img = cv2.imread(img_path)

        # Skip corrupted or unreadable images
        if img is None:
            print(f"⚠ Skipping corrupted image: {img_name}")
            continue

        # Save original image with a prefix
        save_path = os.path.join(output_class_path, f"orig_{count}.png")
        cv2.imwrite(save_path, img)

        original_imgs.append(img)
        count += 1

    # If no original images were successfully loaded, skip augmentation for this class
    if len(original_imgs) == 0:
        print(f"⚠ All images failed to load in {class_name}. Skipping...")
        continue

    # Calculate how many more images are needed to reach TARGET_PER_CLASS
    remaining = TARGET_PER_CLASS - count
    print(f"Generating {remaining} augmented images...")

    # Generate augmented images until TARGET_PER_CLASS is reached
    while count < TARGET_PER_CLASS:
        # Choose a random original image to augment
        img = random.choice(original_imgs).copy()

        # Apply augmentation functions
        img = random_rotation(img)
        img = random_ink_density(img)
        img = slight_thickness_variation(img)

        # Save the augmented image
        save_path = os.path.join(output_class_path, f"aug_{count}.png")
        cv2.imwrite(save_path, img)

        count += 1

print("\n✅ Augmentation Complete!")


Processing class: ක
Generating 38 augmented images...

Processing class: ත
Generating 48 augmented images...

Processing class: ර
Generating 44 augmented images...

Processing class: ල
Generating 46 augmented images...

Processing class: ප
Generating 47 augmented images...

Processing class: ග
Generating 20 augmented images...

Processing class: ස
Generating 43 augmented images...

Processing class: ම
Generating 44 augmented images...

Processing class: හ
Generating 45 augmented images...

Processing class: න
Generating 16 augmented images...

✅ Augmentation Complete!


In [5]:
import glob # Import glob for pattern matching file paths
from tqdm import tqdm # Import tqdm for progress bars

# Create the processed dataset directory if it doesn't exist
os.makedirs(PROCESSED_DIR, exist_ok=True)

total_images = 0

# Iterate through each class (subfolder) in the augmented dataset directory
for class_name in os.listdir(AUGMENTED_DIR):
    class_input_path = os.path.join(AUGMENTED_DIR, class_name)

    # Skip if it's not a directory
    if not os.path.isdir(class_input_path):
        continue

    # Create the output directory for the current class in the processed dataset
    class_output_path = os.path.join(PROCESSED_DIR, class_name)
    os.makedirs(class_output_path, exist_ok=True)

    # Get all image paths in the current class folder
    image_paths = glob.glob(os.path.join(class_input_path, "*.*"))

    print(f"Processing class: {class_name} ({len(image_paths)} images)")

    # Process each image with a progress bar
    for img_path in tqdm(image_paths):
        img = cv2.imread(img_path)

        # Skip corrupted or unreadable images
        if img is None:
            continue

        # Apply the predefined preprocessing function
        processed = img_processing(img)

        # Applt the predifiend crop function
        processed = tight_crop(processed)

        # Resize the processed image to the target IMG_SIZE (128x128)
        processed = cv2.resize(
            processed,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_AREA # Use INTER_AREA for shrinking images
        )

        # Construct the output path and save the processed image
        output_path = os.path.join(
            class_output_path,
            os.path.basename(img_path)
        )

        cv2.imwrite(output_path, processed)
        total_images += 1

print("✅ Done. Total processed images:", total_images)

Processing class: ක (150 images)


100%|██████████| 150/150 [00:12<00:00, 12.33it/s]


Processing class: ත (150 images)


100%|██████████| 150/150 [00:12<00:00, 12.48it/s]


Processing class: ර (150 images)


...

Processing class: හ (150 images)


100%|██████████| 150/150 [00:10<00:00, 14.43it/s]


Processing class: න (150 images)


100%|██████████| 150/150 [00:11<00:00, 13.46it/s]

✅ Done. Total processed images: 1500
